In [2]:
import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Preprocessing and Models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# --- Configuration ---
FEATURES_DATA_DIR = Path("data/03_features")
MODELS_DIR = Path("models")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete. Libraries and paths are ready.")

Setup complete. Libraries and paths are ready.


In [4]:
print("Loading engineered features and metadata...")
loaded_data = joblib.load(FEATURES_DATA_DIR / "features_and_metadata.pkl")
text_embeddings = loaded_data['text_embeddings']
metadata = loaded_data['metadata']
y = loaded_data['target']

# We will now only use the 'CWE' column from metadata.
X_df = pd.concat([
    pd.DataFrame(text_embeddings),
    metadata[['CWE']].reset_index(drop=True)
], axis=1)

# Ensure column names are strings for the pipeline
X_df.columns = X_df.columns.astype(str)

print("Data loaded and prepared.")
X_df.head()

Loading engineered features and metadata...
Data loaded and prepared.


,0,1,2,3,4,5,6,7,8,9,...,375,376,377,378,379,380,381,382,383,CWE
0,-0.033908,0.015571,0.010175,-0.101573,0.007362,0.014034,0.059407,0.035561,-0.019849,0.024568,...,-0.010238,0.026312,-0.115110,0.009623,0.037389,0.039113,0.022055,-0.071572,0.002420,NVD-CWE-noinfo
1,0.006606,0.011746,-0.010097,-0.089709,0.004503,-0.004217,-0.148475,0.039329,0.063173,0.048748,...,-0.004692,0.018525,-0.119607,0.033469,0.012710,0.046535,0.079751,0.002747,-0.060203,CWE-416
2,-0.018710,0.053635,0.025404,-0.103438,0.034413,0.006148,-0.003255,0.045450,-0.025159,-0.027263,...,-0.061399,-0.001974,-0.047818,0.034144,0.022546,0.034643,0.043994,0.055449,-0.104791,CWE-367
3,-0.026057,0.079921,0.044445,-0.124836,0.043646,-0.024125,-0.060651,0.021771,-0.122268,-0.002377,...,-0.125300,0.019577,-0.041202,0.121371,0.117877,0.016177,0.027383,0.017293,0.052080,CWE-755
4,0.005453,0.037343,0.002889,-0.041581,0.016524,0.009072,-0.045070,0.103270,-0.037196,0.027765,...,-0.026399,0.016543,-0.071199,0.040170,0.052186,-0.022264,0.068748,-0.057744,-0.035675,CWE-787


In [5]:
print("Splitting data into training and testing sets...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Splitting data into training and testing sets...
Training set shape: (1905, 385)
Testing set shape: (635, 385)


In [7]:
print("Defining preprocessing pipeline...")
    
categorical_features = ['CWE']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # Keep embedding columns untouched
)

print("Preprocessor defined.")

Defining preprocessing pipeline...
Preprocessor defined.


In [8]:
print("Training the final model... (This may take several minutes)")
    
base_estimators = [
    ('lgbm', lgb.LGBMClassifier(random_state=42)),
    ('xgb', xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'))
]
meta_classifier = LogisticRegression(max_iter=1000)
stacking_classifier = StackingClassifier(
    estimators=base_estimators, final_estimator=meta_classifier, cv=5
)

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', stacking_classifier)])

full_pipeline.fit(X_train, y_train)

print("Model training complete.")

Training the final model... (This may take several minutes)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017043 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97960
[LightGBM] [Info] Number of data points in the train set: 1905, number of used features: 404
[LightGBM] [Info] Start training from score -2.833738
[LightGBM] [Info] Start training from score -1.090769
[LightGBM] [Info] Start training from score -3.086329
[LightGBM] [Info] Start training from score -0.580569
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:30:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97950
[LightGBM] [Info] Number of data points in the train set: 1524, number of used features: 399
[LightGBM] [Info] Start training from score -2.829284
[LightGBM] [Info] Start training from score -1.090769
[LightGBM] [Info] Start training from score -3.094987
[LightGBM] [Info] Start training from score -0.580334
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


KeyboardInterrupt: 

In [ ]:
print("Training the final model... (This may take several minutes)")
    
base_estimators = [
    ('lgbm', lgb.LGBMClassifier(random_state=42)),
    ('xgb', xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'))
]
meta_classifier = LogisticRegression(max_iter=1000)
stacking_classifier = StackingClassifier(
    estimators=base_estimators, final_estimator=meta_classifier, cv=5
)

# Create the full pipeline: preprocess data, then train the model
full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', stacking_classifier)])

full_pipeline.fit(X_train, y_train)

print("Model training complete.")

In [ ]:
print("Generating confusion matrix plot...")
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix for Stacked Ensemble')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig(MODELS_DIR / "best_model_confusion_matrix.png")
plt.show()

In [ ]:
print("Saving the final pipeline...")
joblib.dump(full_pipeline, MODELS_DIR / "full_pipeline.pkl")
joblib.dump(label_encoder, MODELS_DIR / "label_encoder.pkl")

print("\nModel training and evaluation complete!")